In [1]:
# %%
import os
from typing import List
import cv2
import imageio
import tensorflow as tf
from tensorflow.keras import layers, models, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, LearningRateScheduler
from tensorflow.keras.optimizers import Adam
from matplotlib import pyplot as plt
import numpy as np


In [ ]:
vocab = [x for x in "abcdefghijklmnopqrstuvwxyz'?!123456789 "]

char_to_num = tf.keras.layers.StringLookup(vocabulary=vocab, oov_token="")
num_to_char = tf.keras.layers.StringLookup(
    vocabulary=char_to_num.get_vocabulary(), oov_token="", invert=True
)

print(
    f"The vocabulary is: {char_to_num.get_vocabulary()} "
    f"(size ={char_to_num.vocabulary_size()})"
)

def load_video(path:str) -> List[float]: 

    cap = cv2.VideoCapture(path)
    frames = []
    for _ in range(int(cap.get(cv2.CAP_PROP_FRAME_COUNT))): 
        ret, frame = cap.read()
        frame = tf.image.rgb_to_grayscale(frame)
        frames.append(frame[190:236,80:220,:])
    cap.release()
    
    mean = tf.math.reduce_mean(frames)
    std = tf.math.reduce_std(tf.cast(frames, tf.float32))
    return tf.cast((frames - mean), tf.float32) / std

def load_alignments(path:str) -> List[str]: 
    with open(path, 'r') as f: 
        lines = f.readlines() 
    tokens = []
    for line in lines:
        line = line.split()
        if line[2] != 'sil': 
            tokens = [*tokens,' ',line[2]]
    return char_to_num(tf.reshape(tf.strings.unicode_split(tokens, input_encoding='UTF-8'), (-1)))[1:]

def load_data(path: str): 
    path = bytes.decode(path.numpy())
    #file_name = path.split('/')[-1].split('.')[0]
    # File name splitting for windows
    file_name = path.split('\\')[-1].split('.')[0]
    video_path = os.path.join('data','s1',f'{file_name}.mpg')
    alignment_path = os.path.join('data','alignments','s1',f'{file_name}.align')
    frames = load_video(video_path) 
    alignments = load_alignments(alignment_path)
    
    return frames, alignments

def mappable_function(path:str) ->List[str]:
    result = tf.py_function(load_data, [path], (tf.float32, tf.int64))
    return result

The vocabulary is: ['', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', "'", '?', '!', '1', '2', '3', '4', '5', '6', '7', '8', '9', ' '] (size=40)


In [ ]:
data = tf.data.Dataset.list_files('./data/s1/*.mpg')
data = data.shuffle(500, reshuffle_each_iteration=False)
data = data.map(mappable_function)
data = data.padded_batch(2, padded_shapes=([75,None,None,None],[40]))
data = data.prefetch(tf.data.AUTOTUNE)
# Added for split 
train = data.take(450)
test = data.skip(450)

In [ ]:
val = sample.next(); val[0]

InvalidArgumentError: {{function_node __wrapped__IteratorGetNext_output_types_2_device_/job:localhost/replica:0/task:0/device:CPU:0}} ValueError: object __array__ method not producing an array
Traceback (most recent call last):

  File "d:\anaconda3\envs\tf-gpu\lib\site-packages\tensorflow\python\ops\script_ops.py", line 269, in __call__
    return func(device, token, args)

  File "d:\anaconda3\envs\tf-gpu\lib\site-packages\tensorflow\python\ops\script_ops.py", line 147, in __call__
    outputs = self._call(device, args)

  File "d:\anaconda3\envs\tf-gpu\lib\site-packages\tensorflow\python\ops\script_ops.py", line 154, in _call
    ret = self._func(*args)

  File "d:\anaconda3\envs\tf-gpu\lib\site-packages\tensorflow\python\autograph\impl\api.py", line 642, in wrapper
    return func(*args, **kwargs)

  File "C:\Users\qdang\AppData\Local\Temp\ipykernel_4564\4199726544.py", line 41, in load_data
    frames = load_video(video_path)

  File "C:\Users\qdang\AppData\Local\Temp\ipykernel_45

In [ ]:
# %%
# Show first video as GIF
import numpy as np
video = (val[0][0] * 255).astype(np.uint8)  # scale to 0-255
video = np.squeeze(video, axis=-1)
imageio.mimsave('./animation.gif', video, fps=10)

# Show a frame
plt.imshow(val[0][0][35], cmap='gray')
plt.show()

# Decode alignment
tf.strings.reduce_join([num_to_char(word) for word in val[1][0]])


NameError: name 'val' is not defined

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Conv3D, MaxPool3D, BatchNormalization, 
                                     TimeDistributed, GlobalAveragePooling3D,
                                     Bidirectional, LSTM, Dropout, Dense, Input,
                                     Add, LayerNormalization)
from tensorflow.keras.layers import Reshape, Flatten

def build_lipreading_model(input_shape=(75,46,140,1), vocab_size=len(vocab)+1):
    """
    Build an improved 3D-CNN + BiLSTM lip-reading model.
    
    Args:
        input_shape: (frames, height, width, channels)
        vocab_size: number of output characters
    
    Returns:
        Compiled Keras Model
    """
    inputs = Input(shape=input_shape, name='video_input')
    
    # --- 3D CNN Block 1 ---
    x = Conv3D(64, (3,3,3), padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = MaxPool3D((1,2,2))(x)
    
    # --- 3D CNN Block 2 with residual ---
    x_res = x
    x = Conv3D(128, (3,3,3), padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Conv3D(128, (3,3,3), padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    
    # Residual channel match
    x_res = Conv3D(128, (1,1,1), padding='same')(x_res)
    x = Add()([x, x_res])
    x = MaxPool3D((1,2,2))(x)
    
    # --- 3D CNN Block 3 ---
    x = Conv3D(256, (3,3,3), padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPool3D((1,2,2))(x)
    
    # --- Reduce spatial dimensions ---
    x = Conv3D(128, (1,1,1), padding='same', activation='relu')(x)
    x = TimeDistributed(Flatten())(x)

    
    # --- BiLSTM layers with LayerNormalization ---
    x = Bidirectional(LSTM(256, return_sequences=True, kernel_initializer='orthogonal'))(x)
    x = LayerNormalization()(x)
    x = Dropout(0.5)(x)
    
    x = Bidirectional(LSTM(256, return_sequences=True, kernel_initializer='orthogonal'))(x)
    x = LayerNormalization()(x)
    x = Dropout(0.5)(x)
    
    # --- Output layer ---
    outputs = Dense(vocab_size, activation='softmax', kernel_initializer='he_normal', name='char_output')(x)
    
    model = Model(inputs, outputs, name='LipReadingModel')
    return model

# Build model
vocab_size = char_to_num.vocabulary_size() + 1  # +1 for CTC blank
with tf.device('/GPU:0'):
    # Build the model
    model = build_lipreading_model(input_shape=(75,46,140,1), vocab_size=vocab_size)

    # Compile it
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',  # or CTC loss if using sequences
        metrics=['accuracy']
    )

model.summary()


Model: "LipReadingModel"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 video_input (InputLayer)       [(None, 75, 46, 140  0           []                               
                                , 1)]                                                             
                                                                                                  
 conv3d_6 (Conv3D)              (None, 75, 46, 140,  1792        ['video_input[0][0]']            
                                 64)                                                              
                                                                                                  
 batch_normalization_4 (BatchNo  (None, 75, 46, 140,  256        ['conv3d_6[0][0]']               
 rmalization)                    64)                                                

In [ ]:
yhat = model.predict(val[0])

InvalidArgumentError: Graph execution error:

No OpKernel was registered to support Op 'CudnnRNN' used by {{node CudnnRNN}} with these attrs: [seed=0, dropout=0, T=DT_FLOAT, input_mode="linear_input", direction="unidirectional", rnn_mode="lstm", seed2=0, is_training=true]
Registered devices: [CPU, GPU]
Registered kernels:
  <no registered kernels>

	 [[CudnnRNN]]
	 [[LipReadingModel/bidirectional_2/forward_lstm_2/PartitionedCall]] [Op:__inference_predict_function_21473]

In [ ]:
tf.strings.reduce_join([num_to_char(tf.argmax(x)) for x in yhat[1]])

<tf.Tensor: shape=(), dtype=string, numpy=b'ddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddd'>

In [ ]:
model.input_shape

(None, 75, 46, 140, 1)

In [ ]:
model.output_shape

(None, 75, 41)

In [ ]:
def scheduler(epoch, lr):
    if epoch < 30:
        return lr
    else:
        return lr * tf.math.exp(-0.1)

In [ ]:
def CTCLoss(y_true, y_pred):
    batch_len = tf.cast(tf.shape(y_true)[0], dtype="int64")
    input_length = tf.cast(tf.shape(y_pred)[1], dtype="int64")
    label_length = tf.cast(tf.shape(y_true)[1], dtype="int64")

    input_length = input_length * tf.ones(shape=(batch_len, 1), dtype="int64")
    label_length = label_length * tf.ones(shape=(batch_len, 1), dtype="int64")

    loss = tf.keras.backend.ctc_batch_cost(y_true, y_pred, input_length, label_length)
    return loss

In [ ]:
class ProduceExample(tf.keras.callbacks.Callback): 
    def __init__(self, dataset) -> None: 
        self.dataset = dataset.as_numpy_iterator()
    
    def on_epoch_end(self, epoch, logs=None) -> None:
        data = self.dataset.next()
        yhat = self.model.predict(data[0])
        decoded = tf.keras.backend.ctc_decode(yhat, [75,75], greedy=False)[0][0].numpy()
        for x in range(len(yhat)):           
            print('Original:', tf.strings.reduce_join(num_to_char(data[1][x])).numpy().decode('utf-8'))
            print('Prediction:', tf.strings.reduce_join(num_to_char(decoded[x])).numpy().decode('utf-8'))
            print('~'*100)

In [ ]:
model.compile(optimizer=Adam(learning_rate=0.0001), loss=CTCLoss)
checkpoint_callback = ModelCheckpoint(os.path.join('models','checkpoint'), monitor = 'loss', save_weights_only = True)
schedule_callback = LearningRateScheduler(scheduler)
example_callback = ProduceExample(test)

In [ ]:
model.fit(train, validation_data=test, epochs=4, callbacks=[checkpoint_callback, schedule_callback, example_callback])

Epoch 1/4


InvalidArgumentError: Graph execution error:

2 root error(s) found.
  (0) INVALID_ARGUMENT:  ValueError: object __array__ method not producing an array
Traceback (most recent call last):

  File "d:\anaconda3\envs\tf-gpu\lib\site-packages\tensorflow\python\ops\script_ops.py", line 269, in __call__
    return func(device, token, args)

  File "d:\anaconda3\envs\tf-gpu\lib\site-packages\tensorflow\python\ops\script_ops.py", line 147, in __call__
    outputs = self._call(device, args)

  File "d:\anaconda3\envs\tf-gpu\lib\site-packages\tensorflow\python\ops\script_ops.py", line 154, in _call
    ret = self._func(*args)

  File "d:\anaconda3\envs\tf-gpu\lib\site-packages\tensorflow\python\autograph\impl\api.py", line 642, in wrapper
    return func(*args, **kwargs)

  File "C:\Users\qdang\AppData\Local\Temp\ipykernel_4564\1618437690.py", line 41, in load_data
    frames = load_video(video_path)

  File "C:\Users\qdang\AppData\Local\Temp\ipykernel_4564\1618437690.py", line 17, in load_vide